<a href="https://colab.research.google.com/github/juniorrodes/IA-desafio-GA/blob/main/Desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Agente de Monitoramento (Reativo)
O agente de monitoramento deve ser um agente reativo de classificacao de risco. Para ele, sera usado o dataset_m.csv, com o atributo alvo sendo a coluna "risco", com um valor entre 0 (sem risco) e 5.

## 1.1 Carregamento e Exploracao dos Dados

In [ ]:
# Carregar dataset de monitoramento
dfm = pd.read_csv("https://raw.githubusercontent.com/VictorioBF/college-AI-ML/main/desafio1/dataset_m.csv")

print("Shape do dataset:", dfm.shape)
print("\nPrimeiras linhas:")
dfm.head()

In [ ]:
# Informacoes gerais sobre o dataset
print("Tipos de dados:")
print(dfm.dtypes)
print("\nValores nulos:")
print(dfm.isnull().sum())
print("\nEstatisticas descritivas:")
dfm.describe()

In [ ]:
# Distribuicao das classes de risco
print("Distribuicao das classes de risco:")
print(y_counts := dfm['risco'].value_counts().sort_index())

plt.figure(figsize=(8, 4))
dfm['risco'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Distribuicao das Classes de Risco')
plt.xlabel('Nivel de Risco')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 1.2 Pre-processamento

In [ ]:
# Pre-processamento do dataset de monitoramento

# Convertendo datetime e extraindo features temporais
dfm['data'] = pd.to_datetime(dfm['data'])
dfm['hora'] = dfm['data'].dt.hour
dfm['dia_semana'] = dfm['data'].dt.dayofweek
dfm['mes'] = dfm['data'].dt.month

# One-hot encoding para 'terreno' (variavel categorica)
dfm = pd.get_dummies(dfm, columns=['terreno'], prefix='terreno')

# Converter booleano para numerico
dfm['obstaculos'] = dfm['obstaculos'].astype(int)

# Separar features (X) e target (y)
X_m = dfm.drop(['risco', 'data'], axis=1)
y_m = dfm['risco']

print("Features utilizadas:")
print(X_m.columns.tolist())
print(f"\nShape X: {X_m.shape}")
print(f"Shape y: {y_m.shape}")

In [ ]:
# Dividir em treino e teste (80/20)
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42, stratify=y_m
)

print(f"Tamanho treino: {X_m_train.shape[0]}")
print(f"Tamanho teste: {X_m_test.shape[0]}")

# Normalizar features numericas
scaler_m = StandardScaler()
X_m_train_scaled = scaler_m.fit_transform(X_m_train)
X_m_test_scaled = scaler_m.transform(X_m_test)

## 1.3 Treinamento do Modelo

In [ ]:
# Treinar modelo Random Forest para classificacao de risco
modelo_monitoramento = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

modelo_monitoramento.fit(X_m_train_scaled, y_m_train)
print("Modelo de Monitoramento treinado com sucesso!")

## 1.4 Avaliacao do Modelo

In [ ]:
# Predicoes
y_m_pred = modelo_monitoramento.predict(X_m_test_scaled)

# Metricas de avaliacao
acc_m = accuracy_score(y_m_test, y_m_pred)
f1_m = f1_score(y_m_test, y_m_pred, average='weighted')
cm_m = confusion_matrix(y_m_test, y_m_pred)

print("="*60)
print("RESULTADOS - AGENTE DE MONITORAMENTO")
print("="*60)
print(f"\nAcuracia: {acc_m:.4f} ({acc_m*100:.2f}%)")
print(f"F1-Score (weighted): {f1_m:.4f}")
print(f"\nRelatorio de Classificacao:")
print(classification_report(y_m_test, y_m_pred, zero_division=0))

In [ ]:
# Matriz de Confusao
plt.figure(figsize=(8, 6))
sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(y_m.unique()),
            yticklabels=sorted(y_m.unique()))
plt.title('Matriz de Confusao - Agente de Monitoramento')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia das features
feature_importance_m = pd.DataFrame({
    'feature': X_m.columns,
    'importance': modelo_monitoramento.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_m, x='importance', y='feature', color='steelblue')
plt.title('Importancia das Features - Agente de Monitoramento')
plt.xlabel('Importancia')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

---
# Agente de Triagem (Reativo)
O agente de triagem deve ser um agente reativo de classificacao de prioridade. Para ele, sera usado o dataset dataset_t.csv, com o atributo alvo sendo a coluna "prioridade", com valores entre 0 e 8, sendo 8 a prioridade maxima.

## 2.1 Carregamento e Exploracao dos Dados

In [ ]:
# Carregar dataset de triagem
dft = pd.read_csv("https://raw.githubusercontent.com/VictorioBF/college-AI-ML/main/desafio1/dataset_t.csv")

print("Shape do dataset:", dft.shape)
print("\nColunas:")
print(dft.columns.tolist())
print("\nPrimeiras linhas:")
dft.head()

In [ ]:
# Informacoes gerais sobre o dataset
print("Tipos de dados:")
print(dft.dtypes)
print("\nValores nulos:")
print(dft.isnull().sum())
print("\nEstatisticas descritivas:")
dft.describe()

In [ ]:
# Distribuicao das classes de prioridade
print("Distribuicao das classes de prioridade:")
print(dft['prioridade'].value_counts().sort_index())

plt.figure(figsize=(10, 4))
dft['prioridade'].value_counts().sort_index().plot(kind='bar', color='darkorange')
plt.title('Distribuicao das Classes de Prioridade')
plt.xlabel('Nivel de Prioridade')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2.2 Pre-processamento

In [ ]:
# Pre-processamento do dataset de triagem

# Remover coluna vazia/problematica (nivel_aguasentimento parece ser resultado de merge incorreto)
if 'nivel_aguasentimento' in dft.columns:
    dft = dft.drop('nivel_aguasentimento', axis=1)

# Converter datetime e extrair features temporais
dft['data'] = pd.to_datetime(dft['data'])
dft['hora'] = dft['data'].dt.hour
dft['dia_semana'] = dft['data'].dt.dayofweek
dft['mes'] = dft['data'].dt.month

# Converter booleanos para numerico
dft['vulneraveis'] = dft['vulneraveis'].astype(int)
dft['necessita_medico'] = dft['necessita_medico'].astype(int)
dft['confiavel'] = dft['confiavel'].astype(int)

# Encoding para 'origem' (canal de comunicacao)
dft = pd.get_dummies(dft, columns=['origem'], prefix='origem')

# Encoding para 'nivel_agua' (ordinal: tornozelo < cintura < teto)
nivel_agua_map = {'tornozelo': 0, 'cintura': 1, 'teto': 2}
dft['nivel_agua_encoded'] = dft['nivel_agua'].map(nivel_agua_map)

# Encoding para 'sentimento' (ordinal: calmo < nervoso < panico)
sentimento_map = {'calmo': 0, 'nervoso': 1, 'panico': 2}
dft['sentimento_encoded'] = dft['sentimento'].map(sentimento_map)

# Remover colunas originais que ja foram codificadas e a coluna data
dft = dft.drop(['data', 'nivel_agua', 'sentimento'], axis=1)

# Tratar valores nulos restantes
print("Valores nulos apos pre-processamento:")
print(dft.isnull().sum()[dft.isnull().sum() > 0])
dft = dft.fillna(dft.median(numeric_only=True))

# Separar features (X) e target (y)
X_t = dft.drop(['prioridade'], axis=1)
y_t = dft['prioridade']

print(f"\nFeatures utilizadas:")
print(X_t.columns.tolist())
print(f"\nShape X: {X_t.shape}")
print(f"Shape y: {y_t.shape}")

In [ ]:
# Dividir em treino e teste (80/20)
X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t
)

print(f"Tamanho treino: {X_t_train.shape[0]}")
print(f"Tamanho teste: {X_t_test.shape[0]}")

# Normalizar features numericas
scaler_t = StandardScaler()
X_t_train_scaled = scaler_t.fit_transform(X_t_train)
X_t_test_scaled = scaler_t.transform(X_t_test)

## 2.3 Treinamento do Modelo

In [ ]:
# Treinar modelo Random Forest para classificacao de prioridade
modelo_triagem = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

modelo_triagem.fit(X_t_train_scaled, y_t_train)
print("Modelo de Triagem treinado com sucesso!")

## 2.4 Avaliacao do Modelo

In [ ]:
# Predicoes
y_t_pred = modelo_triagem.predict(X_t_test_scaled)

# Metricas de avaliacao
acc_t = accuracy_score(y_t_test, y_t_pred)
f1_t = f1_score(y_t_test, y_t_pred, average='weighted')
cm_t = confusion_matrix(y_t_test, y_t_pred)

print("="*60)
print("RESULTADOS - AGENTE DE TRIAGEM")
print("="*60)
print(f"\nAcuracia: {acc_t:.4f} ({acc_t*100:.2f}%)")
print(f"F1-Score (weighted): {f1_t:.4f}")
print(f"\nRelatorio de Classificacao:")
print(classification_report(y_t_test, y_t_pred, zero_division=0))

In [ ]:
# Matriz de Confusao
plt.figure(figsize=(10, 8))
sns.heatmap(cm_t, annot=True, fmt='d', cmap='Oranges',
            xticklabels=sorted(y_t.unique()),
            yticklabels=sorted(y_t.unique()))
plt.title('Matriz de Confusao - Agente de Triagem')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia das features
feature_importance_t = pd.DataFrame({
    'feature': X_t.columns,
    'importance': modelo_triagem.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance_t, x='importance', y='feature', color='darkorange')
plt.title('Importancia das Features - Agente de Triagem')
plt.xlabel('Importancia')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

---
# Resumo dos Resultados

In [ ]:
print("="*60)
print("RESUMO FINAL DOS MODELOS")
print("="*60)
print(f"\n{'Metrica':<25} {'Monitoramento':<20} {'Triagem':<20}")
print("-"*65)
print(f"{'Acuracia':<25} {acc_m:.4f}{'':<16} {acc_t:.4f}")
print(f"{'F1-Score (weighted)':<25} {f1_m:.4f}{'':<16} {f1_t:.4f}")
print(f"{'Algoritmo':<25} {'Random Forest':<20} {'Random Forest'}")
print(f"{'N. Estimadores':<25} {'200':<20} {'200'}")
print(f"{'Train/Test Split':<25} {'80/20':<20} {'80/20'}")
print("="*60)